# C2 real run on Colab (tasks C2-T6 and C2-T7, addendum A2)

This notebook makes the pre-registered C2 run on a **Colab CPU runtime** (Linux), as addendum A2 fixes:
- Python 3.13.0 and the 13 exact package versions of `c2_requirements.txt`;
- XGBoost with 2 threads per fit;
- the frozen code only.

**Before the 2-3 hour run, a pre-flight check of about 8 minutes:**
1. the bundle is undamaged and holds the frozen code;
2. the environment equals the pinned one;
3. the C2 tests pass;
4. on the made-up noise target, this machine reproduces the Linux reference of the platform test byte for byte:
   - the dry run;
   - the full-size model fingerprints.

If any check fails, the notebook stops **before** the real run and says why. Nothing real has been computed at that
point.

**Steps**
1. Upload the new `c2_colab_bundle.zip` to the top level of *My Drive*. Delete the old one first, so Drive does not
   rename the new one.
2. In the first code cell, `RUN = 'T6'` for the real run. Later, `RUN = 'T7'` for the rerun, in a new session.
3. *Runtime → Change runtime type → CPU* (High-RAM on, if offered), then *Runtime → Run all*. Allow Drive access.
4. With Colab Pro+ background execution you can close the browser tab.
   - If Colab disconnects, open the notebook and *Run all* again. Finished fits are kept on Drive
     (`My Drive/c2_runs/<RUN>/c2x_parts`), so the run resumes where it stopped.
   - Each session repeats the pre-flight check, so a new machine is checked too.
5. At the end the notebook prints the verdict and saves `My Drive/c2_runs/<RUN>_results.zip`. Send that file.

Never reuse a run folder: a finished run is never run again (pre-registration section 13).

In [ ]:
# 1. Settings, the bundle and the checks of its files
RUN = 'T6'          # 'T6' = the real run; 'T7' = its rerun, in a new session (addendum A2.2)
import hashlib, json, os, re, shutil, subprocess, sys, time, zipfile
from google.colab import drive
assert RUN in ('T6', 'T7'), "RUN must be 'T6' or 'T7'"
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
BUNDLE = f'{DRIVE}/c2_colab_bundle.zip'
ROOT = '/content/c2'                         # the unpacked bundle (fresh every session)
PRE = '/content/c2_preflight'                # this session's pre-flight runs (fresh every session)
OUT = f'{DRIVE}/c2_runs/{RUN}'               # results and resume parts, kept on Drive across sessions
for d in (ROOT, PRE):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(ROOT)
with open(f'{ROOT}/bundle_manifest.json') as f:
    man = json.load(f)


def sha(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def rule(path):
    """Pre-registration hash rule: text files after CRLF -> LF, binary files as they are."""
    with open(path, 'rb') as f:
        b = f.read()
    if path.endswith(('.py', '.md', '.csv', '.txt', '.json')):
        b = b.replace(b'\r\n', b'\n')
    return hashlib.sha256(b).hexdigest()


bad = [f for f, h in man['files'].items() if sha(f'{ROOT}/{f}') != h]
assert not bad, f'the bundle was damaged in transfer: upload it again ({bad[:3]})'
FROZEN, IN_BUNDLE = {}, {}
if man.get('frozen'):
    with open(f"{ROOT}/{man['freeze_file']}") as f:
        FROZEN = {p: h for h, p in (l.split() for l in f if l.strip() and not l.startswith('#'))}
    IN_BUNDLE = {p: h for p, h in FROZEN.items() if os.path.isfile(f'{ROOT}/{p}')}   # not this notebook itself
    changed = [p for p, h in IN_BUNDLE.items() if rule(f'{ROOT}/{p}') != h]
    assert not changed, f'frozen files differ from the freeze record: {changed}'
GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
assert not GPU, 'addendum A2: the real run is on a CPU runtime. Runtime -> Change runtime type -> CPU, then Run all'
E1V = f'{ROOT}/Implementation/revision_1/e1_verification'
os.makedirs(OUT, exist_ok=True)
SESSION = 1 + len([d for d in os.listdir(OUT) if d.startswith('session_')])
LOGS = f'{OUT}/session_{SESSION:02d}'        # this session's logs and checks, on Drive
os.makedirs(LOGS)
print(f"{len(man['files'])} files unpacked and undamaged; frozen code: "
      + (f'all {len(IN_BUNDLE)} frozen files in the bundle match the freeze record' if FROZEN
         else 'NO (a real run will be refused)'))
print(f'run {RUN}: results in {OUT}; session {SESSION}')


def run(*args, log):
    """Run a command in e1_verification, show its output live, keep it in LOGS/log, and stop if it fails."""
    args = [str(a) for a in args]
    print('$', ' '.join(args), flush=True)
    p = subprocess.Popen(args, cwd=E1V, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    with open(f'{LOGS}/{log}', 'a') as f:
        f.write('$ ' + ' '.join(args) + '\n')
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
            lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f'step failed (exit {p.returncode}); see the output above')
    return ''.join(lines)

In [ ]:
# 2. A clean Python 3.13.0 environment with the exact package versions (Colab's own Python is not used)
VENV, VPY = '/content/c2venv', '/content/c2venv/bin/python'
shutil.rmtree(VENV, ignore_errors=True)
os.environ.pop('UV_SYSTEM_PYTHON', None)     # Colab sets it; it does not apply to a separate environment
REQ = f"{ROOT}/{man['requirements']}"
run(sys.executable, '-m', 'pip', 'install', '-q', 'uv', log='environment.log')
run(sys.executable, '-m', 'uv', 'venv', '--python', man['python'], VENV, log='environment.log')
run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY, '-r', REQ, log='environment.log')
with open(REQ) as f:
    want = dict(l.strip().split('==') for l in f if l.strip() and not l.startswith('#'))
frozen_env = run(sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', VPY, log='environment.log')
got = dict(l.strip().split('==') for l in frozen_env.splitlines() if '==' in l)
py = run(VPY, '-c', 'import platform; print(platform.python_version())', log='environment.log').strip()
differ = {n: (got.get(n), v) for n, v in want.items() if got.get(n) != v}
print('python', py, '| packages that differ from the pinned versions:', differ or 'none')
assert py == man['python'] and not differ, 'the environment does not match the pinned one'

In [ ]:
# 3. The machine: processors, the threads XGBoost honours, and the layout (2 threads per fit, addendum A2.1)
REMOVED = {v: os.environ.pop(v, None) for v in ('OMP_NUM_THREADS', 'OMP_THREAD_LIMIT')}   # unset for every run
probe = ('import json, c2_xgb_confirm as rn; '
         'print(json.dumps({"machine": rn.machine(), "cpu": rn.cpu_model(), "logical processors": rn.processors(), '
         '"cgroup CPU quota": rn.cgroup_cpus(), "most threads XGBoost honours": rn.xgb_threads(rn.processors())}))')
hw = json.loads(run(VPY, '-c', probe, log='environment.log').strip().splitlines()[-1])
run(VPY, '-c', 'import numpy; numpy.show_runtime()', log='environment.log')
MAXT = hw['most threads XGBoost honours']
assert MAXT >= 2, f'XGBoost can use only {MAXT} thread here; addendum A2 needs 2. Choose a larger CPU runtime'
PROCS = MAXT // 2                            # processes x 2 threads = the threads XGBoost honours
hw.update({'run': RUN, 'session': SESSION, 'processes': PROCS, 'xgboost n_jobs': 2,
           'OpenMP variables removed': REMOVED, 'bundle sha256': sha(BUNDLE), 'frozen': bool(FROZEN)})
T6_VERDICT = f'{DRIVE}/c2_runs/T6/c2x_verdict.json'
if RUN == 'T7' and os.path.isfile(T6_VERDICT):
    with open(T6_VERDICT) as f:
        t6_machines = json.load(f)['provenance']['fits made on (addendum A2.3)']
    hw['T6 made on'] = t6_machines
    same_cpu = all(m.split(' | ')[0] == hw['cpu'] for m in t6_machines)
    print('T7 processor type ' + ('equals' if same_cpu else 'differs from') + ' T6\'s'
          + (' (addendum A2.2 accepts this when Colab assigns no other type)' if same_cpu else ' (as addendum A2.2 asks)'))
with open(f'{LOGS}/environment.json', 'w') as f:
    json.dump(hw, f, indent=1)
print(json.dumps(hw, indent=1))

In [ ]:
# 4. The C2 test suites (leakage, rules, runner); no real-target test-year prediction is made
for t in ('test_c2x.py', 'test_c2x_verdict.py', 'test_c2_runner.py'):
    run(VPY, f'../tests/{t}', log='tests.log')

In [ ]:
# 5. Pre-flight check 1 (made-up target): the dry run must equal the Linux reference of the platform test byte for byte
REPORT = []
run(VPY, 'c2x_features.py', log='preflight.log')                  # the feature digests of this machine


def digests(path):
    with open(path) as f:
        return dict(re.findall(r'^(BASE|meta digest|  \S+)\s.*?([0-9a-f]{16})$', f.read(), re.M))


mine = digests(f'{E1V}/outputs/c2_t2_features_summary.txt')
shutil.copy(f'{E1V}/outputs/c2_t2_features_summary.txt', f'{LOGS}/c2_t2_features_summary.txt')
like = [k for k, p in man['linux_features_summaries'].items() if digests(f'{ROOT}/{p}') == mine]
REPORT.append('inputs: BASE and meta digests equal the reference: '
              + str(all(mine[k] == digests(f"{ROOT}/{next(iter(man['linux_features_summaries'].values()))}")[k]
                        for k in ('BASE', 'meta digest')))
              + '; weather digests equal those of: ' + (', '.join(like) or 'neither reference machine (last-bit '
                'logarithm differences; information only, the model check below decides)'))
REF = f"{ROOT}/{man['linux_reference']['dry']}"
run(VPY, 'c2_xgb_confirm.py', '--dry-run', '--seeds', 0, 1, '--processes', PROCS, '--n-jobs', 2,
    '--out', f'{PRE}/dry', log='preflight.log')


def content(path):
    """The verdict without what is expected to differ: the provenance block (machine, code hashes) and the A2 hash,
    which the reference (made before addendum A2) does not carry."""
    with open(path) as f:
        v = json.load(f)
    v.pop('provenance')
    v.pop('addendum_A2_sha256', None)
    return v


same = {f: rule(f'{PRE}/dry/{f}') == rule(f'{REF}/{f}') for f in ('dry_pooled.csv', 'dry_city.csv')}
same['dry_predictions.npz'] = sha(f'{PRE}/dry/dry_predictions.npz') == sha(f'{REF}/dry_predictions.npz')
same['verdict apart from provenance'] = content(f'{PRE}/dry/dry_verdict.json') == content(f'{REF}/dry_verdict.json')
with open(f'{PRE}/dry/dry_verdict.json') as f:
    fits_on = json.load(f)['provenance']['fits made on (addendum A2.3)']
same['all 64 fits recorded on this machine'] = fits_on == {hw['machine']: 64}
REPORT.append(f'pre-flight dry run ({PROCS} processes x 2 threads) against the Linux reference: {same}')
print('\n'.join(REPORT))
with open(f'{LOGS}/preflight.txt', 'w') as f:
    f.write('\n'.join(REPORT) + '\n')
assert all(same.values()), ('PRE-FLIGHT FAILED: this machine does not reproduce the Linux reference. The real run is '
                            f'not started. Send the folder {LOGS}')

In [ ]:
# 6. Pre-flight check 2 (made-up target): full-size models (400 trees) must equal the Linux reference fingerprints
run(VPY, 'c2_xgb_confirm.py', '--timing', '--processes', PROCS, '--n-jobs', 2, '--out', f'{PRE}/timing',
    log='preflight.log')
with open(f'{PRE}/timing/fingerprints_cpu_p{PROCS}_t2.json') as f:
    mine_fp = json.load(f)['arms']
with open(f"{ROOT}/{man['linux_reference']['fingerprints']}") as f:
    ref_fp = json.load(f)['arms']
n_same = sum(mine_fp.get(a) == ref_fp[a] for a in ref_fp)
with open(f'{PRE}/timing/run_log.log') as f:
    timing = [l.split(' ', 2)[2].strip() for l in f if 'TIMING' in l][-1]
REPORT += [f'pre-flight full-size models: identical to the Linux reference in {n_same} of {len(ref_fp)} arms', timing]
print('\n'.join(REPORT[-2:]))
with open(f'{LOGS}/preflight.txt', 'w') as f:
    f.write('\n'.join(REPORT) + '\n')
assert n_same == len(ref_fp), ('PRE-FLIGHT FAILED: full-size models differ from the Linux reference. The real run is '
                               f'not started. Send the folder {LOGS}')
print('PRE-FLIGHT PASSED')

In [ ]:
# 7. The real run: 32 arms x 40 seeds = 1,280 fits, resumable from the parts on Drive
if os.path.isfile(f'{OUT}/c2x_verdict.json'):
    print(f'{RUN} finished in an earlier session; it is not run again (pre-registration section 13)')
else:
    assert FROZEN, 'this bundle is not from the frozen code: no real run (tracker ground rule 3)'
    t0 = time.time()
    run(VPY, 'c2_xgb_confirm.py', '--processes', PROCS, '--n-jobs', 2, '--out', OUT, log='run.log')
    print(f'this session: {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# 8. The verdict, the T7-against-T6 comparison, and the results file for download
with open(f'{OUT}/c2x_verdict.txt') as f:
    print(f.read())
T6 = f'{DRIVE}/c2_runs/T6'
if RUN == 'T7' and os.path.isfile(f'{T6}/c2x_verdict.json'):
    def strip(path):
        with open(path) as f:
            v = json.load(f)
        v.pop('provenance')
        return v
    cmp = {f: rule(f'{OUT}/{f}') == rule(f'{T6}/{f}') for f in ('c2x_pooled.csv', 'c2x_city.csv')}
    cmp['c2x_predictions.npz'] = sha(f'{OUT}/c2x_predictions.npz') == sha(f'{T6}/c2x_predictions.npz')
    cmp['verdict apart from provenance'] = strip(f'{OUT}/c2x_verdict.json') == strip(f'{T6}/c2x_verdict.json')
    line = f'T7 against T6 (addendum A2.2): {cmp}'
    print(line)
    with open(f'{LOGS}/t7_vs_t6.txt', 'w') as f:
        f.write(line + '\n')
zpath = f'{DRIVE}/c2_runs/{RUN}_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base, dirs, files in os.walk(OUT):
        dirs[:] = sorted(d for d in dirs if d != 'c2x_parts')    # the parts are inside c2x_predictions.npz
        for f in sorted(files):
            p = os.path.join(base, f)
            zf.write(p, os.path.relpath(p, OUT))
drive.flush_and_unmount()
print('saved', zpath, '- send this file')